# WMaxTwin for Tecator Functional Regression, R notebook

This notebook reproduces the Tecator functional-regression example in R. It mirrors the Python Jupyter notebook but uses a standalone base-R implementation. The split is over whole spectra, and wavelet features are computed within each spectrum.

## Core geometry

Ordinary MaxTwin is the endpoint $\gamma=0$. Nested WMaxTwin uses

$$D_\gamma^2(i,i')=(1-\gamma)D_0^2(i,i')+\gamma D_w^2(i,i'),$$

where $D_0$ is raw functional distance and $D_w$ is the calibration-weighted Haar wavelet distance.

In [ ]:
source("wmaxtwin_tecator_functional_regression_R.R")


## Load Tecator data

The package includes the StatLib Tecator file. The protocol uses samples 1--172 for calibration and 173--215 for external testing.

In [ ]:
download_tecator(DATA_PATH)
dat <- load_tecator(DATA_PATH)
ip <- interp_to_power2(dat$X, dat$wavelengths, m = 128)
X128 <- ip$X
wl128 <- ip$grid
y_fat <- dat$endpoints[, "fat"]
cat("Spectra:", nrow(X128), " Grid points:", ncol(X128), "\n")


## Run the study

For a quick GitHub run, `R_REPS <- 8`. Increase to 50 or 100 for a fuller Monte Carlo experiment.

In [ ]:
R_REPS <- 8
ans <- run_study(R = R_REPS, seed = 20260619)
summary_table <- ans$summary
weights_table <- data.frame(scale_j = as.integer(names(ans$weights)), weight = as.numeric(ans$weights))


## Calibration-only scale weights

Weights are average squared correlations between Haar coefficients and fat response, computed using the 172 calibration samples only.

In [ ]:
print(weights_table, row.names = FALSE)


## Split comparison table

In [ ]:
print(summary_table, row.names = FALSE)


## Figures

In [ ]:
knitr::include_graphics(file.path(ans$outdir, "fig1_tecator_spectra.png"))


In [ ]:
knitr::include_graphics(file.path(ans$outdir, "fig2_quartile_mean_spectra.png"))


In [ ]:
knitr::include_graphics(file.path(ans$outdir, "fig3_response_scale_weights.png"))


In [ ]:
knitr::include_graphics(file.path(ans$outdir, "fig4_validation_curves_rep0.png"))


In [ ]:
knitr::include_graphics(file.path(ans$outdir, "fig5_test_rmse_boxplot.png"))


In [ ]:
knitr::include_graphics(file.path(ans$outdir, "fig6_selected_jmax_boxplot.png"))


In [ ]:
knitr::include_graphics(file.path(ans$outdir, "fig7_selected_alpha_boxplot.png"))


In [ ]:
knitr::include_graphics(file.path(ans$outdir, "fig8_gamma_path_rmse.png"))


## Interpretation

The diagnostic is the nested path: MaxTwin is $\gamma=0$, while positive $\gamma$ values add scale-weighted wavelet geometry. The goal is not universal dominance but a scientifically meaningful perturbation of MaxTwin when the response depends on multiresolution spectral structure.